In [1]:
!pip install pandas openpyxl

In [2]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd

In [ ]:
df = pd.read_excel("../data/Online Retail.xlsx")

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.dtypes

In [ ]:
df.info()

In [ ]:
df.isnull()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
df[df["CustomerID"].isnull()]["Quantity"].describe()

In [ ]:
df[df["Quantity"] < 0].head(20)

In [ ]:
df["InvoiceNo"].astype(str).str.startswith("C").sum()

In [ ]:
(df["InvoiceNo"].astype(str).str.startswith("C").mean()) * 100

In [ ]:
df[df["Quantity"] == -80995]

In [ ]:
df["InvoiceNo"].astype(str).str[0].value_counts()

In [ ]:
df[~df["InvoiceNo"].astype(str).str.isnumeric()]["InvoiceNo"].unique()[:50]

In [ ]:
df[df["InvoiceNo"].astype(str).str.startswith("C")]["Quantity"].describe()

In [ ]:
df[
    (df["Quantity"] < 0) &
    (~df["InvoiceNo"].astype(str).str.startswith("C"))
]["InvoiceNo"].head(20)

In [ ]:
df[df["InvoiceNo"] == "536589"]

In [ ]:
(df["Quantity"] < 0).sum()

In [ ]:
(df["InvoiceNo"].astype(str).str.startswith("C")).sum()

In [ ]:
df[df["InvoiceNo"].astype(str).str.startswith("A")].shape

In [ ]:
df[df["InvoiceNo"].astype(str).str.startswith("A")].head(20)

In [ ]:
df[df["InvoiceNo"].astype(str).str.startswith("A")]["Quantity"].describe()

In [ ]:
df[
    (df["Quantity"] < 0) &
    (~df["InvoiceNo"].astype(str).str.startswith("C")) &
    (~df["InvoiceNo"].astype(str).str.startswith("A"))
].head(20)

In [ ]:
#  Find negative-quantity transactions that do not use the C prefix, to investigate whether they are another special transaction type.
df[
    (df["Quantity"] < 0) &
    (~df["InvoiceNo"].astype(str).str.startswith("C")) &
    (~df["InvoiceNo"].astype(str).str.startswith("A"))
].shape

In [ ]:
#  Find negative-quantity transactions that do not use the C prefix, to investigate whether they are another special transaction type.
df[
    (df["Quantity"] < 0) &
    (~df["InvoiceNo"].astype(str).str.startswith("C")) &
    (~df["InvoiceNo"].astype(str).str.startswith("A"))
]["UnitPrice"].value_counts()

In [ ]:
#  Count unique customers available in the raw dataset.
df["CustomerID"].nunique()

In [ ]:
#  Count transactions that do not have a CustomerID.
df["CustomerID"].isnull().sum()

In [ ]:
#  Remove transactions without a CustomerID because customer-level modeling requires a known customer identifier.
#removed transactions where CustomerID
df_clean = df.dropna(subset=["CustomerID"])

In [ ]:
#  Count unique customers available in the raw dataset.
df_clean["CustomerID"].nunique()

In [ ]:
#  Check the size of the cleaned dataset.
df_clean.shape

In [ ]:
#  Remove C-prefixed transactions because the analysis identified them as cancellation/return-type records rather than normal purchases.
#Since all the C-prefixed transactions had negative quantities, indicating cancellation/return-type transactions. We removed these aswell
df_clean = df_clean[
    ~df_clean["InvoiceNo"].astype(str).str.startswith("C")
]

In [ ]:
# Check the size of the cleaned dataset.
df_clean.shape

In [ ]:
#  Remove C-prefixed transactions because the analysis identified them as cancellation/return-type records rather than normal purchases.
df_clean["InvoiceNo"].astype(str).str.startswith("C").sum()

In [ ]:
#  Inspect A-prefixed invoices to understand what type of accounting records they represent.
# A-Prefixed were accounting adjustments rather than customer purchases and they had no CustomerID so they were already 
# removed when we removed missing CustomerIDs

df_clean[df_clean["InvoiceNo"].astype(str).str.startswith("A")]

In [ ]:
#  Verify that no negative-quantity transactions remain after the cleaning rules are applied.
df_clean[df_clean["Quantity"] < 0].shape

In [ ]:
# Save the cleaned transaction-level dataset as a CSV file for reuse.
#Saved the cleaned dataset in a CSV
df_clean.to_csv("../data/clean_transactions.csv", index=False)

In [ ]:
#  Preview the cleaned transaction data.
df_clean.head()

In [ ]:
# Check the size of the cleaned dataset.
df_clean.shape

In [ ]:
#  Find the earliest transaction date in the cleaned dataset.
#checking the transaction dates for cleaned dataset
df_clean["InvoiceDate"].min()

In [ ]:
#  : Find the latest transaction date in the cleaned dataset.
df_clean["InvoiceDate"].max()

In [ ]:
# Find the latest transaction date in the cleaned dataset.
#cutoff date - Since we need 30 days of future data to determine whether someone purchased again.
#We have to move 30 days backward 

cutoff_date = df_clean["InvoiceDate"].max() - pd.Timedelta(days=30)

cutoff_date

In [ ]:
# Create the feature period: transactions on or before the cutoff date are used to calculate customer behavior features.
#separating the clean data into the feature period and the target period.
feature_data = df_clean[df_clean["InvoiceDate"] <= cutoff_date]

In [ ]:
# Create the target period: transactions after the cutoff date are used to determine whether customers returned.
target_data = df_clean[df_clean["InvoiceDate"] > cutoff_date]

In [ ]:
# Check how many transactions are available in the feature period.
feature_data.shape

In [ ]:
# Check how many transactions are available in the target period.
target_data.shape

In [ ]:
# Verify the date range covered by the feature period.
feature_data["InvoiceDate"].min(), feature_data["InvoiceDate"].max()

In [ ]:
# Verify the date range covered by the target period.
target_data["InvoiceDate"].min(), target_data["InvoiceDate"].max()

# Feature Engineering

Customer-level feature engineering for the retention prediction model (Zain's contribution). This starts from the cleaned transaction data produced in the data-cleaning step and builds the RFM + behavioral feature set used to train the model.

In [ ]:
import pandas as pd

In [ ]:
# Load the cleaned transactions produced by the data-cleaning step
df_clean = pd.read_csv("../Data/clean_transactions.csv", parse_dates=["InvoiceDate"])
df_clean.shape

## Define the feature and target time windows

We need 30 days of future data to determine whether a customer purchased again, so the cutoff date is moved 30 days back from the last transaction.

In [ ]:
cutoff_date = df_clean["InvoiceDate"].max() - pd.Timedelta(days=30)
cutoff_date

In [ ]:
# Separating the clean data into the feature period and the target period
feature_data = df_clean[df_clean["InvoiceDate"] <= cutoff_date]
target_data = df_clean[df_clean["InvoiceDate"] > cutoff_date]

feature_data.shape, target_data.shape

**Feature Engineering**

Since the dataset was transactional, but our model needs customer-level information.

In [ ]:
# Feature 1: Recency
last_purchase = feature_data.groupby("CustomerID")["InvoiceDate"].max()
recency = (cutoff_date - last_purchase).dt.days
recency.head()

In [ ]:
# Feature 2: Frequency
frequency = feature_data.groupby("CustomerID")["InvoiceNo"].nunique()
frequency.head()

In [ ]:
# Feature 3: Monetary
feature_data["TotalAmount"] = feature_data["Quantity"] * feature_data["UnitPrice"]

# Calculate Monetary per customer
monetary = feature_data.groupby("CustomerID")["TotalAmount"].sum()
monetary.head()

In [ ]:
rfm = pd.concat([recency, frequency, monetary], axis=1)
rfm.columns = ["Recency", "Frequency", "Monetary"]
rfm.head()

In [ ]:
# Feature 4: Total Quantity
total_quantity = feature_data.groupby("CustomerID")["Quantity"].sum()
total_quantity.head()

In [ ]:
# Feature 5: Unique Products
unique_products = feature_data.groupby("CustomerID")["StockCode"].nunique()
rfm["UniqueProducts"] = unique_products
rfm.head()

In [ ]:
# Feature 6: Average Order Value
rfm["AverageOrderValue"] = rfm["Monetary"] / rfm["Frequency"]
rfm.head()

In [ ]:
# Feature 7: Purchase Days
feature_data["PurchaseDate"] = feature_data["InvoiceDate"].dt.date
unique_purchase_days = feature_data.groupby("CustomerID")["PurchaseDate"].nunique()
rfm["UniquePurchaseDays"] = unique_purchase_days
rfm.head()

In [ ]:
# Feature 8: Customer Lifespan
first_purchase = feature_data.groupby("CustomerID")["InvoiceDate"].min()
customer_lifespan = (last_purchase - first_purchase).dt.days
rfm["CustomerLifespan"] = customer_lifespan
rfm.head()

In [ ]:
# Feature 9: Orders in the last 30 days of the feature window
last_30_days_start = cutoff_date - pd.Timedelta(days=30)

orders_last_30_days = (
    feature_data[feature_data["InvoiceDate"] > last_30_days_start]
    .groupby("CustomerID")["InvoiceNo"]
    .nunique()
)

rfm["OrdersLast30Days"] = orders_last_30_days
rfm["OrdersLast30Days"] = rfm["OrdersLast30Days"].fillna(0).astype(int)
rfm.head()

In [ ]:
# Target: did the customer purchase again in the 30-day target window?
repeat_customers = target_data["CustomerID"].unique()
rfm["Target"] = rfm.index.isin(repeat_customers).astype(int)
rfm["Target"].value_counts()

## Validate the engineered feature set

In [ ]:
rfm.shape

In [ ]:
rfm.isnull().sum()

In [ ]:
rfm.dtypes

In [ ]:
rfm.describe().T

## Save engineered features

Handed off as a customer-level feature table for the modeling step.

In [ ]:
rfm.to_csv("../Data/rfm_features.csv")